In [ ]:
%load_ext autoreload
%autoreload 2

import uproot
import awkward as ak

import matplotlib.pylab as plt
import numpy as np

import time

from hist import Hist

import babar_analysis_tools as bat

from analysis_variables import *

import myPIDselector

import pandas as pd
import seaborn as sns

import ROOT
import pdf_definitions as pdfs

This one is tricky, because there are two options.

First, we need to define the signal window, something like 5.27 < m_es < 5.285 GeV (you can fine tune).

Then you have two options:

1) Define the "bkg" region as 5.27 < m_ES < 5.285 GeV in the BKG MC. Then the relative size is the MC/data lumi ratio (i.e. the weight you need to multiply to rescale the MC to the data). There are 1 or 3 events in that window (this is difficult to say from the plots) and one signal event. I need to know the MC/data lumi ratio to do the calculation.

2) Define the bkg region as 5.20 < m_ES <5.27 GeV in the DATA. Then the relative window size is 0.215 (=0.015/0.07). There are two events in the bkg window. In that case, TRolke gives
TRolke tr
tr.SetCL(0.9);
tr.SetPoissonBkgGaussEff(1,2,0.35,0.215,0.02)  /* N_sig, N_bkg, efficiency, relative window size, sigma_efficiency (my guess for illustration)
tr.GetUpperLimit() = 1.8 event

If the background is well described by the MC, then I would go with option 1 since m_ES has a different shape in the bkg and sig regions (it dips in the signal region). If the MC is less than optimal, then switch to option 2. In any case, I would compare the two methods (but you won't be able to claim a blind analysis)

Hope this helps

# Read in the data

In [ ]:
BNC_tag = ""
BNC_bool = False
#UNBLINDED_tag = ""
UNBLINDED_tag = "_UNBLINDED"
#ntrain_tag = 'nsig_20000_nbkg_20000'

#BNC_tag = "_BNC"
#BNC_bool = True
#ntrain_tag = 'nsig_30000_nbkg_30000'
#ntrain_tag = 'nsig_40000_nbkg_40000_trial0'
#ntrain_tag = 'nsig_40000_nbkg_40000_trial1'
#ntrain_tag = 'features_2_nsig_40000_nbkg_40000_trial0'

# WHAT WE USE
ntrain_tag = 'features_4_nsig_30000_nbkg_30000_trial15'

# Playing around
#ntrain_tag = 'features_8_nsig_30000_nbkg_30000_trial15'


# Read in the dfs
infilename_sp = f"DATAFRAME_SP_MODEL_MLPClassifier_CUTS_1_2_3_{ntrain_tag}{BNC_tag}.pkl"

#infilename_col = f"DATAFRAME_COL_MODEL_MLPClassifier_CUTS_1_2_3_{ntrain_tag}{BNC_tag}.pkl"
infilename_col = f"DATAFRAME_COL_MODEL_MLPClassifier_CUTS_1_2_3_{ntrain_tag}{BNC_tag}{UNBLINDED_tag}.pkl"

df_sp = pd.read_parquet(infilename_sp)
df_col = pd.read_parquet(infilename_col)


########### FOM ############
df_fom = bat.punzi_fom_nn(df_sp, df_col, region_definitions=region_definitions, BNC=BNC_bool, sigma=4.0)

fom_max = df_fom['fom'].max()

print(fom_max)

filter = df_fom['fom'] == fom_max

df_fom[filter]
###########################
print()
max_proba_cut = df_fom[filter]['thresh'].values[0]
print(f'max_proba_cut: {max_proba_cut}')
sig_eff_from_ML = df_fom[filter]['sig_pct'].values[0]
print(f'sig_eff_from_ML: {sig_eff_from_ML}')

# From the Btatuplemaker
#sig_eff *= 0.5963
# This is only due to the cut on the ML variable
print(f'sig_eff_from_ML: {sig_eff_from_ML}')

#proba_cut = 0.82


In [ ]:
df_fom['sig_pct']

In [ ]:
# BNV
save_dir = './BNV_pLambda_plots/'
print(f'{max_proba_cut = }')

fig, axes = plt.subplots(3,1, sharex=True, figsize=(8,8))

labels = ['SP - bkg', 'SP - sig', 'Collision data']

for i in range(0,3):

    idx = None
    spmode = None
    df_tmp = None
    
    if i==0:

        # Use them all
        mask = (~df_sp['used_in_bkg_train']) | (df_sp['used_in_bkg_train'])
        spmode = '998'
        df_tmp = df_sp[mask]

    elif i==1:

        mask = (~df_sp['used_in_sig_train'])
        spmode = '-999'
        df_tmp = df_sp[mask]
    
    elif i==2:
        spmode = '0'
        df_tmp = df_col
        mask = np.ones(len(df_tmp), dtype=bool)
    
    spmask = (df_tmp['spmode']==spmode)
    if i==0:# Background
        spmask = (df_tmp['spmode']!='-999')
    
    mask = mask &  (df_tmp['cut_-1']==True)
    if BNC_bool:
        print("Making BNC cuts")
        mask = mask & (df_tmp['cut_2']==True) & (df_tmp['cut_3']==True)  & (df_tmp['cut_4']==True)

    mask = mask & (df_tmp['proba'] > max_proba_cut)
    
    mask = mask & (df_tmp['BpostFitDeltaE']<0.05) & (df_tmp['BpostFitDeltaE']>-0.05)

    #var = 'proba'
    var = 'BpostFitMes'

    #plt.subplot(3,1,i+1)
    df_tmp[spmask & mask][var].hist(bins=50, range=(5.2,5.3), label=labels[i], ax=axes[i])#, range=(0,0.99))
    axes[i].legend()

    # Print!
    mask_region = (df_tmp['BpostFitMes']>5.2) & (df_tmp['BpostFitMes']<5.27)
    nnorm = len(df_tmp[spmask & mask & mask_region][var])

    mask_region = (df_tmp['BpostFitMes']>=5.27)
    nsig = len(df_tmp[spmask & mask & mask_region][var])

    print(f"# norm: {nnorm}    # sig: {nsig}")
    
    #print(df_tmp[spmask & mask][var])
    
axes[2].set_xlabel(r'$M_{ES}$ (GeV/c$^2$)', fontsize=18)

plt.tight_layout()

tag = "UPPER_LIMIT_CALCS"
plt.savefig(f'{save_dir}/mes_tight_de_probcut_{max_proba_cut:.2f}_{tag}{BNC_tag}{UNBLINDED_tag}.png')

In [ ]:
mask = (df_sp['cut_2']==True) & (df_sp['cut_3']==True) & (df_sp['cut_4']==True)
mask = mask & (df_sp['spmode']=='998')
df_sp[mask]['used_in_bkg_train'].value_counts()

# RooFit

In [ ]:
import ROOT
from ROOT import RooFit
import numpy as np

# %jsroot on   # uncomment for interactive inline canvases

xlo, xhi, nbins = 5.2, 5.3, 50

# Observable. RooFit auto-appends the unit to axis titles.
mes = ROOT.RooRealVar('BpostFitMes', 'M_{ES}', xlo, xhi, 'GeV/c^{2}')
mes.setBins(nbins)

def make_dataset(name, vals, var=mes, xlo=xlo, xhi=xhi):
    """Build an unbinned RooDataSet from a numpy array of values."""
    vals = np.asarray(vals, dtype='float64')
    vals = vals[(vals >= xlo) & (vals <= xhi)]   # also drops NaNs
    # ROOT >= 6.26:
    return ROOT.RooDataSet.from_numpy({var.GetName(): vals}, [var])

    # --- Fallback for older ROOT (no from_numpy): bin into a TH1 -> RooDataHist ---
    # h = ROOT.TH1F(name, name, nbins, xlo, xhi); h.Sumw2(False)
    # h.FillN(len(vals), vals, np.ones(len(vals)))
    # return ROOT.RooDataHist(name, name, ROOT.RooArgList(var), h)

In [ ]:
save_dir = './BNV_pLambda_plots/'
print(f'{max_proba_cut = }')

labels = ['MC - bkg', 'MC - sig', 'Collision data']

c = ROOT.TCanvas('c_mes', 'Mes', 800, 800)
c.Divide(1, 3)
c.SetTopMargin(0)

ROOT.gStyle.SetOptTitle(0)

keepalive = []  # PyROOT GC will blank pads if frames/legends aren't held

for i in range(0, 3):
    idx = None
    spmode = None
    df_tmp = None

    if i == 0:
        mask = (~df_sp['used_in_bkg_train']) | (df_sp['used_in_bkg_train'])
        spmode = '998'
        df_tmp = df_sp[mask]
    elif i == 1:
        mask = (~df_sp['used_in_sig_train'])
        spmode = '-999'
        df_tmp = df_sp[mask]
    elif i == 2:
        spmode = '0'
        df_tmp = df_col
        mask = np.ones(len(df_tmp), dtype=bool)

    spmask = (df_tmp['spmode'] == spmode)
    if i == 0:  # Background
        spmask = (df_tmp['spmode'] != '-999')

    mask = mask & (df_tmp['cut_-1'] == True)
    if BNC_bool:
        print("Making BNC cuts")
        mask = mask & (df_tmp['cut_2'] == True) & (df_tmp['cut_3'] == True) & (df_tmp['cut_4'] == True)

    mask = mask & (df_tmp['proba'] > max_proba_cut)
    mask = mask & (df_tmp['BpostFitDeltaE'] < 0.05) & (df_tmp['BpostFitDeltaE'] > -0.05)

    var = 'BpostFitMes'

    # --- selected values -> RooDataSet ---
    vals = df_tmp[spmask & mask][var].to_numpy()
    ds = make_dataset(f'ds_{i}', vals)

    # --- frame + Poisson error bars ---
    frame = mes.frame(RooFit.Bins(nbins))
    ds.plotOn(frame,
              RooFit.DataError(ROOT.RooAbsData.Poisson),  # asymmetric Poisson intervals
              RooFit.XErrorSize(0),                       # drop horizontal bars (HEP convention)
              RooFit.MarkerStyle(20), RooFit.MarkerSize(1.2),
              RooFit.Name('data'))

    c.cd(i + 1)

    ROOT.gPad.SetLeftMargin(0.13)
    ROOT.gPad.SetRightMargin(0.04)            # <-- stops 5.30 from being clipped
    ROOT.gPad.SetBottomMargin(0.26)
    ROOT.gPad.SetTopMargin(0.03)

    
    frame.GetXaxis().SetLabelSize(0.06)
    frame.GetYaxis().SetLabelSize(0.06)

    #frame.GetXaxis().SetTitleSize(20)
    frame.GetYaxis().SetTitleSize(0.06)

    frame.Draw()

    #leg = ROOT.TLegend(0.10, 0.78, 0.38, 0.90)
    #leg.AddEntry(frame.findObject('data'), labels[i], 'lep')
    #leg.SetBorderSize(0); leg.SetFillStyle(0)
    #leg.Draw()

    pave = None
    if i==0 or i==1:
        pave = ROOT.TPaveText(0.22, 0.82, 0.53, 0.92, 'NDC')
    elif i==2:
        pave = ROOT.TPaveText(0.32, 0.82, 0.63, 0.92, 'NDC')

    pave.AddText(labels[i])
    pave.SetTextSize(0.10)
    pave.SetFillStyle(0)   # transparent; use 1001 + SetFillColor for solid
    pave.SetBorderSize(0)                          # 0 for no border
    pave.Draw()
    keepalive += [ds, frame, pave]
    
    #if i < 2:
    #    frame.GetXaxis().SetTitle('')   # only label the bottom pad (sharex-style)
    #else:
    #    frame.GetXaxis().SetTitle('M_{ES} (GeV/c^{2})')
    #    frame.GetXaxis().SetTitleSize(0.06)

    frame.GetXaxis().SetTitle('M_{ES} (GeV/c^{2})')
    frame.GetXaxis().SetTitleSize(0.10)

    
    keepalive += [ds, frame]

    # --- yields, same as before ---
    mask_region = (df_tmp['BpostFitMes'] > 5.2) & (df_tmp['BpostFitMes'] < 5.27)
    nnorm = len(df_tmp[spmask & mask & mask_region][var])
    mask_region = (df_tmp['BpostFitMes'] >= 5.27)
    nsig = len(df_tmp[spmask & mask & mask_region][var])
    print(f"# norm: {nnorm}    # sig: {nsig}")

c.Update()

tag = "UPPER_LIMIT_CALCS"
c.SaveAs(f'{save_dir}/mes_tight_de_probcut_{max_proba_cut:.2f}_{tag}{BNC_tag}{UNBLINDED_tag}_ROOT.png')

c.Draw()  # inline display in the notebook

In [ ]:
decay = 'BNV'

proba_cut = max_proba_cut

total_efficiencies, total_efficiency_errs, conv_factors, conv_factor_errs = bat.calculate_conversion_factor(df_sp, df_col, \
                                                                 decay=decay, \
                                                                 region_definitions=region_definitions, \
                                                                 sig_eff_after_ML=sig_eff_from_ML)

print()
print(f'{conv_factors[0] = :20.5f}')
print(f'{conv_factor_errs[0] = :16.5f}')
print(f'{total_efficiencies[0] = :20.5f}')
print(f'{total_efficiency_errs[0] = :16.5f}')

In [ ]:
fig,axes = plt.subplots(1,3, figsize=(12,4))

# BNV
proba_cut = max_proba_cut
#proba_cut = 0.0

if BNC_bool:
    proba_cut = max_cut
    #proba_cut = 0.90

deloline, dehiline = -0.05, 0.05

#de_cut = 0.07
de_cut = 0.2

# SP bkg
mask = (df_sp['spmode'] != '-999')

# Not used in training
mask = mask & (~df_sp['used_in_bkg_train'])# | (df_sp['used_in_bkg_train'])


if BNC_bool:
    mask = mask &  (df_sp['cut_2']==True) & (df_sp['cut_3']==True)  & (df_sp['cut_4']==True)
else:
    mask = mask &  (df_sp['cut_-1']==True)

mask = mask & (df_sp['BpostFitMes']>5.20)# & (df_sp['BpostFitDeltaE']>-0.07)

mask = mask & (df_sp['BpostFitDeltaE']<de_cut) & (df_sp['BpostFitDeltaE']>-de_cut)

mask = mask & (df_sp['proba'] > proba_cut)

df_sp[mask & (df_sp['spmode']=='998')].plot.scatter(x='BpostFitMes', y='BpostFitDeltaE', ax=axes[0])#, label='SP-998')#, label='SP')
df_sp[mask & (df_sp['spmode']=='1005')].plot.scatter(x='BpostFitMes', y='BpostFitDeltaE', ax=axes[0], c='orange')#, label='SP-1005')#, label='SP')

axes[0].plot([5.2, 5.29], [deloline, deloline], 'r--', lw=3)
axes[0].plot([5.2, 5.29], [dehiline, dehiline], 'r--', lw=3)
#plt.legend()
axes[0].set_title(f'Bkg SP (NN > {proba_cut:.2f})')

# SP sig
mask = (df_sp['spmode'] == '-999')

if BNC_bool:
    mask = mask &  (df_sp['cut_2']==True) & (df_sp['cut_3']==True)  & (df_sp['cut_4']==True)
else:
    mask = mask &  (df_sp['cut_-1']==True)

mask = mask & (df_sp['BpostFitMes']>5.20)# & (df_sp['BpostFitDeltaE']>-0.07)

mask = mask & (df_sp['BpostFitDeltaE']<de_cut) & (df_sp['BpostFitDeltaE']>-de_cut)

mask = mask & (df_sp['proba'] > proba_cut)


df_sp[mask].plot.scatter(x='BpostFitMes', y='BpostFitDeltaE', ax=axes[1], s=0.1, alpha=0.1)#, label='SP')
axes[1].plot([5.2, 5.29], [deloline, deloline], 'r--', lw=3)
axes[1].plot([5.2, 5.29], [dehiline, dehiline], 'r--', lw=3)
axes[1].set_ylim(-0.2, 0.2)
#plt.legend()
axes[1].set_title(f'Sig SP (NN > {proba_cut:.2f})')


# Data
mask = (df_col['spmode'] == '0')

if BNC_bool:
    mask = mask &  (df_col['cut_2']==True) & (df_col['cut_3']==True)  & (df_col['cut_4']==True)
else:
    mask = mask &  (df_col['cut_-1']==True)


mask = mask & (df_col['BpostFitMes']>5.20)# & (df_sp['BpostFitDeltaE']>-0.07)

mask = mask & (df_col['BpostFitDeltaE']<de_cut) & (df_col['BpostFitDeltaE']>-de_cut)

mask = mask & (df_col['proba'] > proba_cut)


df_col[mask].plot.scatter(x='BpostFitMes', y='BpostFitDeltaE', ax=axes[2])#, label='Collision data')
axes[2].plot([5.2, 5.29], [deloline, deloline], 'r--', lw=3)
axes[2].plot([5.2, 5.29], [dehiline, dehiline], 'r--', lw=3)
axes[2].set_ylim(-0.2, 0.2)
#plt.legend()
axes[2].set_title(f'Collision data (NN > {proba_cut:.2f})')

plt.tight_layout()

plt.savefig(f'{save_dir}/sp_and_collision_de_vs_mes_probcut_{proba_cut:.2f}_{tag}{BNC_tag}{UNBLINDED_tag}.png')

mask_de = (df_col['BpostFitDeltaE']<0.05) & (df_col['BpostFitDeltaE']>-0.05)
df_col[mask & mask_de]['BpostFitMes'].values

In [ ]:
import ROOT
import numpy as np

ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)

# axis ranges (BpostFitMes has only a lower cut, so pick a sensible upper edge)
xlo, xhi = 5.20, 5.30
ylo, yhi = -0.20, 0.20
deloline, dehiline = -0.05, 0.05

def tgraph_from(df_sel, color=ROOT.kBlue, mstyle=20, msize=0.6):
    x = df_sel['BpostFitMes'].to_numpy(dtype='float64')
    y = df_sel['BpostFitDeltaE'].to_numpy(dtype='float64')
    g = ROOT.TGraph(len(x), x, y) if len(x) else ROOT.TGraph()
    g.SetMarkerColor(color); g.SetMarkerStyle(mstyle); g.SetMarkerSize(msize)
    return g

def th2_from(df_sel, name, nx=80, ny=80):
    h = ROOT.TH2F(name, '', nx, xlo, xhi, ny, ylo, yhi)
    x = df_sel['BpostFitMes'].to_numpy(dtype='float64')
    y = df_sel['BpostFitDeltaE'].to_numpy(dtype='float64')
    if len(x):
        w = np.ones(len(x), dtype='float64')
        h.FillN(len(x), x, y, w)
    h.SetStats(0)
    return h

def dashed_lines(x1=5.20, x2=5.29):
    out = []
    for yv in (deloline, dehiline):
        l = ROOT.TLine(x1, yv, x2, yv)
        l.SetLineColor(ROOT.kRed); l.SetLineStyle(2); l.SetLineWidth(3)
        out.append(l)
    return out

In [ ]:
# BNV
proba_cut = max_proba_cut
if BNC_bool:
    proba_cut = max_cut

de_cut = 0.2

c = ROOT.TCanvas('c_scatter', 'de vs mes', 1200, 400)
c.Divide(3, 1)
keepalive = []

# ---------- Panel 0 : Bkg SP (two species overlaid) ----------
mask = (df_sp['spmode'] != '-999')
mask = mask & (~df_sp['used_in_bkg_train'])
if BNC_bool:
    mask = mask & (df_sp['cut_2'] == True) & (df_sp['cut_3'] == True) & (df_sp['cut_4'] == True)
else:
    mask = mask & (df_sp['cut_-1'] == True)
mask = mask & (df_sp['BpostFitMes'] > 5.20)
mask = mask & (df_sp['BpostFitDeltaE'] < de_cut) & (df_sp['BpostFitDeltaE'] > -de_cut)
mask = mask & (df_sp['proba'] > proba_cut)

g998  = tgraph_from(df_sp[mask & (df_sp['spmode'] == '998')],  color=ROOT.kBlue, mstyle=20, msize=1.2)
g1005 = tgraph_from(df_sp[mask & (df_sp['spmode'] == '1005')], color=ROOT.kOrange + 1, mstyle=21, msize=1.2)

c.cd(1)
fr0 = ROOT.gPad.DrawFrame(xlo, ylo, xhi, yhi)

ROOT.gPad.SetLeftMargin(0.18)
ROOT.gPad.SetRightMargin(0.04)            # <-- stops 5.30 from being clipped
ROOT.gPad.SetBottomMargin(0.20)
ROOT.gPad.SetTopMargin(0.03)


fr0.SetTitle(f'Bkg SP (NN > {proba_cut:.2f});M_{{ES}} (GeV/c^{{2}});#DeltaE (GeV)')
g998.Draw('P SAME')
g1005.Draw('P SAME')
lines0 = dashed_lines()
for l in lines0: l.Draw()
leg0 = ROOT.TLegend(0.25, 0.82, 0.45, 0.96)
leg0.AddEntry(g998, 'MC (light-quark)', 'p')
leg0.AddEntry(g1005, 'MC (charm)', 'p')
leg0.SetBorderSize(0); leg0.SetFillStyle(0); leg0.Draw()
leg0.SetTextSize(0.05)

fr0.GetXaxis().SetTitle('M_{ES} (GeV/c^{2})')
fr0.GetYaxis().SetTitle('#Delta E (GeV)')
fr0.GetYaxis().SetTitleOffset(1.5)         # distance of title from axis; lower = closer in

fr0.GetXaxis().SetNdivisions(505)   # ~5 primary divisions, optimized to round numbers
fr0.GetXaxis().SetLabelSize(0.05)
fr0.GetXaxis().SetTitleSize(0.05)

fr0.GetYaxis().SetLabelSize(0.05)
fr0.GetYaxis().SetTitleSize(0.06)


keepalive += [fr0, g998, g1005, leg0, lines0]

# ---------- Panel 1 : Sig SP (high stats -> TH2 COLZ) ----------
mask = (df_sp['spmode'] == '-999')
if BNC_bool:
    mask = mask & (df_sp['cut_2'] == True) & (df_sp['cut_3'] == True) & (df_sp['cut_4'] == True)
else:
    mask = mask & (df_sp['cut_-1'] == True)
mask = mask & (df_sp['BpostFitMes'] > 5.20)
mask = mask & (df_sp['BpostFitDeltaE'] < de_cut) & (df_sp['BpostFitDeltaE'] > -de_cut)
mask = mask & (df_sp['proba'] > proba_cut)

h_sig = th2_from(df_sp[mask], 'h_sig')
h_sig.SetTitle(f'Sig SP (NN > {proba_cut:.2f});M_{{ES}} (GeV/c^{{2}});#DeltaE (GeV)')

c.cd(2)
ROOT.gPad.SetRightMargin(0.15)   # room for the z color bar
ROOT.gPad.SetLeftMargin(0.18)
ROOT.gPad.SetRightMargin(0.04)            # <-- stops 5.30 from being clipped
ROOT.gPad.SetBottomMargin(0.20)
ROOT.gPad.SetTopMargin(0.03)


# ROOT.gPad.SetLogz()            # uncomment if the peak swamps the tails
h_sig.Draw('COLZ')
lines1 = dashed_lines()
for l in lines1: l.Draw()
keepalive += [h_sig, lines1]

# ---------- Panel 2 : Collision data (scatter) ----------
mask = (df_col['spmode'] == '0')
if BNC_bool:
    mask = mask & (df_col['cut_2'] == True) & (df_col['cut_3'] == True) & (df_col['cut_4'] == True)
else:
    mask = mask & (df_col['cut_-1'] == True)
mask = mask & (df_col['BpostFitMes'] > 5.20)
mask = mask & (df_col['BpostFitDeltaE'] < de_cut) & (df_col['BpostFitDeltaE'] > -de_cut)
mask = mask & (df_col['proba'] > proba_cut)

h_sig.GetXaxis().SetTitle('M_{ES} (GeV/c^{2})')
h_sig.GetYaxis().SetTitle('#Delta E (GeV)')
h_sig.GetYaxis().SetTitleOffset(1.5)         # distance of title from axis; lower = closer in

h_sig.GetXaxis().SetNdivisions(505)   # ~5 primary divisions, optimized to round numbers
h_sig.GetXaxis().SetLabelSize(0.05)
h_sig.GetXaxis().SetTitleSize(0.05)

h_sig.GetYaxis().SetLabelSize(0.05)
h_sig.GetYaxis().SetTitleSize(0.06)




g_data = tgraph_from(df_col[mask], color=ROOT.kBlack, msize=1.2)

c.cd(3)
fr2 = ROOT.gPad.DrawFrame(xlo, ylo, xhi, yhi)
ROOT.gPad.SetLeftMargin(0.18)
ROOT.gPad.SetRightMargin(0.04)            # <-- stops 5.30 from being clipped
ROOT.gPad.SetBottomMargin(0.20)
ROOT.gPad.SetTopMargin(0.03)

fr2.SetTitle(f'Collision data (NN > {proba_cut:.2f});M_{{ES}} (GeV/c^{{2}});#DeltaE (GeV)')
g_data.Draw('P SAME')

fr2.GetXaxis().SetTitle('M_{ES} (GeV/c^{2})')
fr2.GetYaxis().SetTitle('#Delta E (GeV)')
fr2.GetYaxis().SetTitleOffset(1.5)         # distance of title from axis; lower = closer in

fr2.GetXaxis().SetNdivisions(505)   # ~5 primary divisions, optimized to round numbers
fr2.GetXaxis().SetLabelSize(0.05)
fr2.GetXaxis().SetTitleSize(0.05)

fr2.GetYaxis().SetLabelSize(0.05)
fr2.GetYaxis().SetTitleSize(0.06)


lines2 = dashed_lines()
for l in lines2: l.Draw()
keepalive += [fr2, g_data, lines2]

c.Update()
c.SaveAs(f'{save_dir}/sp_and_collision_de_vs_mes_probcut_{proba_cut:.2f}_{tag}{BNC_tag}{UNBLINDED_tag}_ROOT.png')
c.Draw()

# same final extraction as before
mask_de = (df_col['BpostFitDeltaE'] < 0.05) & (df_col['BpostFitDeltaE'] > -0.05)
df_col[mask & mask_de]['BpostFitMes'].values

In [ ]:
print(repr(fr0.GetYaxis().GetTitle()))   # should print '#DeltaE (GeV)'


In [ ]:
dataset_information= pd.read_csv("dataset_statistics.csv")
cs_data= pd.read_csv("SP_cross_sections_and_labels.csv")

no_notes= cs_data.drop(["Uncertainty","Note: cross sections found at https://babar-wiki.heprc.uvic.ca/bbr_wiki/index.php/Physics/Cross_sections,_luminosities,_and_other_vital_stats"], axis= 1)
no_notes


In [ ]:
cs_data

In [ ]:
bkg_spmodes= ["998","1005","3981","1235","1237"]
sig_spmodes= ["-999"]

spmodes= bkg_spmodes#+sig_spmodes

weights= {}
for sp in spmodes: 
    weights[sp]= bat.scaling_value(int(sp),dataset_information=dataset_information, cs_data= cs_data, plot= False, verbose= False)

weights

# #1

1) Define the "bkg" region as 5.27 < m_ES < 5.285 GeV in the BKG MC. Then the relative size is the MC/data lumi ratio (i.e. the weight you need to multiply to rescale the MC to the data). There are 1 or 3 events in that window (this is difficult to say from the plots) and one signal event. I need to know the MC/data lumi ratio to do the calculation.

In [ ]:
# SP bkg
mask = (df_sp['spmode'] != '-999')
# Not used in training
mask = mask & (~df_sp['used_in_bkg_train'])# | (df_sp['used_in_bkg_train'])

if BNC_bool:
    mask = mask &  (df_sp['cut_2']==True) & (df_sp['cut_3']==True)  & (df_sp['cut_4']==True)
else:
    mask = mask &  (df_sp['cut_-1']==True)

mask = mask & (df_sp['BpostFitDeltaE']<0.05) & (df_sp['BpostFitDeltaE']>-0.05)
mask = mask & (df_sp['proba'] > max_proba_cut)


mask_sig = (df_sp['BpostFitMes']>5.27) & (df_sp['BpostFitMes']<=5.285)
mask_sb  = (df_sp['BpostFitMes']<=5.27)

events_sb = df_sp[mask & mask_sb]['BpostFitMes']
events_sig = df_sp[mask & mask_sig]['BpostFitMes']

print(events_sb)
print(events_sig)

print()

wt = weights['998']

n_sp_sb  = len(events_sb)
n_sp_sig = len(events_sig)

#n_bkg_sp = n_bkg_sp_org

print(f'{wt = :.3f}')
print(f'{n_sp_sig = }')
print(f'{n_sp_sb = }')


# #2 `TRolke`

https://root.cern.ch/root/html522/tutorials/math/Rolke.C.html

https://arxiv.org/pdf/0907.3450

In [ ]:
# ASSUME ML efficiency is 1
decay = 'BNV'

proba_cut = max_proba_cut

total_efficiencies, total_efficiency_errs, conv_factors, conv_factor_errs = bat.calculate_conversion_factor(df_sp, df_col, \
                                                                 decay=decay, \
                                                                 region_definitions=region_definitions, \
                                                                 sig_eff_after_ML=1)

print()
print(f'{conv_factors[0] = :20.5f}')
print(f'{conv_factor_errs[0] = :16.5f}')
print(f'{total_efficiencies[0] = :20.5f}')
print(f'{total_efficiency_errs[0] = :16.5f}')

In [ ]:
# Use proper sig eff
decay = 'BNV'

proba_cut = max_proba_cut

print(f"sig efficiency due to ML: {sig_eff_from_ML:.5f}")

total_efficiencies, total_efficiency_errs, conv_factors, conv_factor_errs = bat.calculate_conversion_factor(df_sp, df_col, \
                                                                 decay=decay, \
                                                                 region_definitions=region_definitions, \
                                                                 sig_eff_after_ML=sig_eff_from_ML)



print()
print(f'{conv_factors[0] = :20.5f}')
print(f'{conv_factor_errs[0] = :16.5f}')
print(f'{total_efficiencies[0] = :20.5f}')
print(f'{total_efficiency_errs[0] = :16.5f}')

In [ ]:
tr = ROOT.TRolke()


In [ ]:
#eff = 1.0 # Signal efficiency for testing
eff = total_efficiencies[0] # Total signal efficiencies
eff_relative_uncertainty = total_efficiency_errs[0]
eff_uncertainty = eff * eff_relative_uncertainty

print(f'{wt = :.4f}')
print()
print(f'{eff = :.4f}')
print(f'{eff_uncertainty = :.4f}')
print()

###########################################################
# Using MC
#print("Using MC to estimate area")
#N_sb = n_sp_sb # THIS IS WHAT IS OBSERVED IN THE SIDEBAND AND MUST BE AN INTEGER
## wt is a fraction (e.g. 0.25) so this number should be how much *more* MC there is than data.
#tau_mc_data = 1.0/wt

###########################################################
# Using data
print("Using data to estimate area")
N_sb  = 3 # THIS IS WHAT IS OBSERVED IN THE SIDEBAND AND MUST BE AN INTEGER
tau_mc_data = 1 

N_sig = 1       # THIS IS WHAT IS OBSERVED IN THE SIGNAL REGION IN THE DATA 


print(f'{N_sig = }')
print(f'{N_sb = }')
print()

# We think this should be sideband area divided by signal area
tau_area = (5.27 - 5.2)/(5.285 - 5.27)
#tau_area = 1/tau_area # DON'T USE THIS, FOR TESTING
tau = tau_area * tau_mc_data

print(f"{tau_area = :9.3f}")
print(f"{tau_mc_data = :6.3f}")
print(f"{tau = :14.3f}")

tr.SetCL(0.9);
tr.SetPoissonBkgGaussEff(N_sig, N_sb, eff, tau, eff_uncertainty)  # N_sig, N_bkg, efficiency, relative window size, sigma_efficiency (my guess for illustration)
upper_limit = tr.GetUpperLimit()# = 1.8 event

#tr.CalculateInterval()

print(f"\n{upper_limit = :.2f}")

f = conv_factors[0]

print(f"{f = :.4e}")

# The conversion factor has the efficiency baked in, so we need to back it out.
br = upper_limit*eff / f

print(f"\n{br = :.4e}")


In [ ]:
tr.GetLowerLimit()

In [ ]:
#tr.GetCriticalNumber([1,2,3], -1)

In [ ]:
# Sanity check
NB = 470e6
ndecay = br*NB

print(f"{ndecay = }")
      
brlam = 0.6 
ndecay *= brlam
print(f"{ndecay = }")

ndecay *= eff
print(f"{ndecay = }")

# Comments from Frank

"I think using TRolke to obtain the confidence interval is reasonable.
It would take some Monte Carlo work to really check coverage (as would
any other approach), but I think it is good enough. 

You have a shape for the
background (I think you trust it based on its agreement with data even for higher
statistics). Then the normalization is provided by how many counts you see in the
sideband. I think this is expected to be something like 7. That is pretty good. You
can put a term in the likelihood to let that fluctuate (or using TRolke with background = Poisson, as I understand things).

For the significance, I think you want to compute a p-value.
Your null hypothesis is no signal, and the alternative is signal > 0.
The p value is the probability, assuming the null hypothesis, that the signal
you extract will be as large, or larger, than what you actually observe.

You will need to do a simulation to compute this, generating many experiments
according to the null hypothesis and running your fit on each experiment. This gives you the distribution of the signal under the null hypothesis. 

From this, you look at how
probable your actual observation is.

Since you know the shape, the main issue could turn out to be how well you know
the background rate. If you see very few background events then your knowledge of
the null hypothesis may be uncertain if you are using data to estimate it. An alternative
is to use the MC to estimate it if necessary. 
"

In [ ]:
nsp_bkg = 7

scale = 0.25

ndata = 5

nest_bkg = nsp_bkg * scale

nest_bkg * 2 # Because it is often 1/2 of the data